In [12]:
# Core libraries
#%pip install -U pandas numpy scikit-learn umap-learn hdbscan plotly

# BERTopic + embeddings
#%pip install -U bertopic sentence-transformers

### Imports and helper functions


In [13]:
import re
import pandas as pd

from pathlib import Path
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP
from hdbscan import HDBSCAN

from collections import Counter
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

In [14]:
try:
    from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
    HAS_REPR = True
except Exception:
    HAS_REPR = False

# c-TF-IDF transformer is optional
try:
    from bertopic.vectorizers import ClassTfidfTransformer
    HAS_CTFIDF = True
except Exception:
    HAS_CTFIDF = False


def clean_text(x) -> str:
    """Basic cleanup: normalize whitespace, remove newlines."""
    if pd.isna(x):
        return ""
    s = str(x).replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def build_document(row: pd.Series) -> str:
    """Build one document (string) from Title + Abstract + AuthorKeywords."""
    title = clean_text(row.get("Title", ""))
    abstract = clean_text(row.get("Abstract", ""))
    keywords = clean_text(row.get("AuthorKeywords", ""))
    parts = [p for p in [title, abstract, keywords] if p]
    return " . ".join(parts)


def slugify(s: str) -> str:
    """Turn a label into a URL/file safe slug."""
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", "-", s).strip("-")
    return s


### Configuration

Set paths and BERTopic hyperparameters.

- `nr_topics`: force the model down to a target number of clusters/topics (optional).
- `min_topic_size`: minimum cluster size (higher = fewer, larger clusters).


In [15]:
# ---- Paths ----
INPUT_CSV = "../data/processed/dataset_clean.csv"   
OUTPUT_DIR = "../data/processed"

# ---- Model settings ----
NR_TOPICS = None          # set to None to skip topic reduction: 30
MIN_TOPIC_SIZE = 15
LANGUAGE = "english"

# ---- Filtering settings ----
MIN_DOC_LEN = 30        # drop docs shorter than this (helps reduce -1 outliers)

# Make sure output dir exists
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

print("Output folder:", out_dir.resolve())


Output folder: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/data/processed


### Load dataset and build documents
- read the CSV
- optionally drop duplicate DOIs
- build `docs` (one string per paper)
- drop empty/too-short documents


In [16]:
df = pd.read_csv(INPUT_CSV, encoding="utf-8", encoding_errors="replace")
print("Rows loaded:", len(df))
print("Columns:", list(df.columns))

# Optional: remove duplicates by DOI if present
if "DOI" in df.columns:
    before = len(df)
    df = df.drop_duplicates(subset=["DOI"], keep="first").reset_index(drop=True)
    print(f"Dropped {before - len(df)} duplicate DOIs")

# Build documents for topic modeling
docs = df.apply(build_document, axis=1).tolist()

# Drop empty/too-short docs (these inflate the -1 outlier topic)
keep = [i for i, d in enumerate(docs) if len(d) >= MIN_DOC_LEN]
df = df.iloc[keep].reset_index(drop=True)
docs = [docs[i] for i in keep]

print("Rows after filtering:", len(df))


Rows loaded: 3531
Columns: ['Conference', 'Year', 'Title', 'DOI', 'PaperType', 'Abstract', 'AuthorNames-Deduped', 'AuthorAffiliation', 'InternalReferences', 'AuthorKeywords', 'AminerCitationCount', 'CitationCount_CrossRef', 'PubsCited_CrossRef', 'Downloads_Xplore', 'Award', 'GraphicsReplicabilityStamp']
Dropped 0 duplicate DOIs
Rows after filtering: 3531


### Build BERTopic model

Key choices:
- **Embeddings**: `all-mpnet-base-v2` (higher quality than MiniLM; slower but usually better topics)
- **Stopwords**: English + domain-specific stopwords (improves interpretability)
- **UMAP**: reduce embedding dimensionality before clustering
- **HDBSCAN**: density-based clustering; creates an outlier topic `-1`
- Optional:
  - improved c-TF-IDF (BM25 weighting)
  - improved representation labels (KeyBERT-inspired + MMR)


In [17]:
# Better embeddings (more accurate than MiniLM; slower but worth it)
# If speed matters, switch to: SentenceTransformer("all-MiniLM-L6-v2")
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# Domain stopwords to improve interpretability
domain_stop = {
    "visualization", "visualizations", "visualize", "visualizing", "visual",
    "analytics", "analysis", "approach", "method", "methods", "technique",
    "system", "framework", "model", "models", "data", "dataset", "datasets",
    "interactive", "interaction", "user", "users", "paper", "results",
    "using", "based", "task", "tasks", "provide", "propose", "present",
}
stop_words = set(ENGLISH_STOP_WORDS).union(domain_stop)

vectorizer_model = CountVectorizer(
    stop_words=list(stop_words) if LANGUAGE == "english" else None,
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.6,
)

# Optional: better c-TF-IDF weighting
ctfidf_model = None
if HAS_CTFIDF:
    ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

# Better manifold + clustering for text corpora
umap_model = UMAP(
    n_neighbors=25,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_TOPIC_SIZE,
    min_samples=max(2, MIN_TOPIC_SIZE // 3),
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

# Topic representation / labeling (optional)
representation_model = None
if HAS_REPR:
    representation_model = [
        KeyBERTInspired(),
        MaximalMarginalRelevance(diversity=0.4),
    ]

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    min_topic_size=MIN_TOPIC_SIZE,
    calculate_probabilities=True,
    language=LANGUAGE,
    verbose=True,
)


### Fit BERTopic and improve outliers

- fit the model
- optionally **re-assign outliers** (topic `-1`) into existing topics
- optionally reduce to a target number of topics (`NR_TOPICS`)


In [18]:
topics, probs = topic_model.fit_transform(docs)

# Re-assign many -1 outliers into existing topics (often a big improvement)
try:
    topics = topic_model.reduce_outliers(docs, topics, probabilities=probs, threshold=0.05)
    topic_model.update_topics(
        docs,
        topics=topics,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model
    )
    print("Outlier reduction applied.")
except Exception as e:
    print("Outlier reduction skipped:", e)

# Optional: force a target number of topics after fitting
if NR_TOPICS is not None:
    topic_model.reduce_topics(docs, nr_topics=NR_TOPICS)
    topics = topic_model.topics_

print("Unique topics (including -1):", len(set(topics)))


2026-01-05 17:18:56,813 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/111 [00:00<?, ?it/s]

2026-01-05 17:22:19,357 - BERTopic - Embedding - Completed ✓
2026-01-05 17:22:19,441 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-05 17:22:34,216 - BERTopic - Dimensionality - Completed ✓
2026-01-05 17:22:34,312 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-05 17:22:34,899 - BERTopic - Cluster - Completed ✓
2026-01-05 17:22:34,937 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-05 17:23:16,477 - BERTopic - Representation - Completed ✓
100%|██████████| 1/1 [00:01<00:00,  1.86s/it]
2026-01-05 17:23:19,017 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outlier reduction applied.
Unique topics (including -1): 54


### Create human-readable cluster names (topic labels)

We try BERTopic's `generate_topic_labels`. 


In [19]:
topic_info = topic_model.get_topic_info()

# Generate readable labels (version-dependent)
try:
    labels = topic_model.generate_topic_labels(nr_words=4, topic_prefix=False, separator=" • ")
    try:
        topic_model.set_topic_labels(labels)  # optional (version-dependent)
    except Exception:
        pass
except Exception:
    labels = None

# Build a label map
if labels is not None:
    label_map = dict(zip(topic_info["Topic"].tolist(), labels))
else:
    label_map = {}
    for t in topic_info["Topic"].tolist():
        if t == -1:
            label_map[t] = "Other / Mixed"
            continue
        words = [w for (w, _) in topic_model.get_topic(t)[:6]]
        label_map[t] = " • ".join(words[:4])

topic_info["Category"] = topic_info["Topic"].map(label_map)
topic_info.head(10)


,Topic,Count,Name,Representation,Representative_Docs,Category
0,0,247,0_unstructured meshes_triangle meshes_mesh sim...,"[unstructured meshes, triangle meshes, mesh si...",[Out-of-core isosurface extraction of time-var...,unstructured meshes • triangle meshes • mesh s...
1,1,239,1_flow field_flow fields_computational fluid d...,"[flow field, flow fields, computational fluid ...",[Vector Field Topology of Time-Dependent Flows...,flow field • flow fields • computational fluid...
2,2,183,2_multidimensional scaling_scatterplots_contin...,"[multidimensional scaling, scatterplots, conti...",[Revisiting Dimensionality Reduction Technique...,multidimensional scaling • scatterplots • cont...
3,3,147,3_graph layouts_node link diagrams_link diagra...,"[graph layouts, node link diagrams, link diagr...",[Divided Edge Bundling for Directional Network...,graph layouts • node link diagrams • link diag...
4,4,113,4_graphical representations_dataflow_declarati...,"[graphical representations, dataflow, declarat...",[Declarative Language Design for Interactive V...,graphical representations • dataflow • declara...
5,5,92,5_bar charts_uncertainty_chart_uncertainties,"[bar charts, uncertainty, chart, uncertainties...",[Examining Effort in 1D Uncertainty Communicat...,bar charts • uncertainty • chart • uncertainties
6,6,107,6_direct volume rendering_volume rendering_vol...,"[direct volume rendering, volume rendering, vo...",[Extinction-Based Shading and Illumination in ...,direct volume rendering • volume rendering • v...
7,7,69,7_weather forecasting_weather prediction_weath...,"[weather forecasting, weather prediction, weat...",[An Interactive Framework for Visualization of...,weather forecasting • weather prediction • wea...
8,8,86,8_explainable machine learning_deep neural net...,"[explainable machine learning, deep neural net...",[NeuroCartography: Scalable Automatic Visual S...,explainable machine learning • deep neural net...
9,9,80,9_sensemaking process_sensemaking_provenance_s...,"[sensemaking process, sensemaking, provenance,...",[Supporting Communication and Coordination in ...,sensemaking process • sensemaking • provenance...


### Add cluster number + cluster name back into the original dataset

This creates a new CSV that contains your original columns plus:

- `Cluster` (topic number)
- `ClusterName` (human-readable topic label)
- `ClusterSlug` (slugified label)

This matches your request: **"add in this dataset only number of cluster"** (and we also add the name).


In [20]:
# Document-level outputs from BERTopic
doc_info = topic_model.get_document_info(docs, df=df)

# Add friendly label columns
doc_info["ClusterName"] = doc_info["Topic"].map(label_map)
doc_info["ClusterSlug"] = doc_info["ClusterName"].map(slugify)

# Create a clean dataset copy with cluster columns
df_with_clusters = df.copy()
df_with_clusters["Cluster"] = doc_info["Topic"].values
df_with_clusters["ClusterName"] = doc_info["ClusterName"].values
df_with_clusters["ClusterSlug"] = doc_info["ClusterSlug"].values

### Group ~30 clusters into 10 *macro categories* 


In [22]:
N_MACRO = 10  # <-- change this if you want a different number of macro categories

# Use only real topics (exclude -1 outliers) to build macro groups
topic_ids = topic_info.loc[topic_info.Topic != -1, "Topic"].astype(int).tolist()

# Topic embeddings aligned to topic ids
topic_emb = topic_model.topic_embeddings_[topic_ids]

# Normalize for cosine-like clustering
topic_emb_norm = normalize(topic_emb)

# Cluster the topic vectors into macro groups
kmeans = KMeans(n_clusters=N_MACRO, random_state=42, n_init="auto")
macro_ids = kmeans.fit_predict(topic_emb_norm)

# Map: topic -> macro_id
macro_map = dict(zip(topic_ids, macro_ids))

def macro_label(macro_id: int, n_words: int = 4, per_topic_words: int = 10) -> str:
    """Create a quick macro label from the most frequent top-words across member topics."""
    member_topics = [t for t in topic_ids if macro_map[t] == macro_id]
    c = Counter()
    for t in member_topics:
        for w, _ in topic_model.get_topic(t)[:per_topic_words]:
            c[w] += 1
    return " • ".join([w for w, _ in c.most_common(n_words)])

macro_label_map = {m: macro_label(m) for m in sorted(set(macro_ids))}

# Topic -> MacroName (string)
topic_to_macro_name = {t: f"Macro {macro_map[t]}: {macro_label_map[macro_map[t]]}" for t in topic_ids}
topic_to_macro_name[-1] = "Macro Other: Outliers / Mixed"

# Add macro columns to topic_info
topic_info["MacroId"] = topic_info["Topic"].map(lambda t: macro_map.get(int(t), -1))
topic_info["MacroName"] = topic_info["Topic"].map(lambda t: topic_to_macro_name.get(int(t), "Macro Other: Outliers / Mixed"))

# Add macro columns to the dataset (df_with_clusters)
df_with_clusters["MacroId"] = df_with_clusters["Cluster"].map(lambda t: macro_map.get(int(t), -1) if pd.notna(t) else -1)
df_with_clusters["MacroName"] = df_with_clusters["Cluster"].map(lambda t: topic_to_macro_name.get(int(t), "Macro Other: Outliers / Mixed") if pd.notna(t) else "Macro Other: Outliers / Mixed")

# Save new outputs
df_with_clusters.to_csv(out_dir / "output/dataset_with_clusters_and_macros.csv", index=False)
topic_info.to_csv(out_dir / "output/topic_summary_with_macros.csv", index=False)

# Also write a macro summary table
macro_summary = (
    topic_info[topic_info.Topic != -1]
    .groupby(["MacroId", "MacroName"], as_index=False)["Count"].sum()
    .sort_values("Count", ascending=False)
)

macro_summary.to_csv(out_dir / "output/macro_summary.csv", index=False)


macro_summary.head(10)


,MacroId,MacroName,Count
2,2,Macro 2: medical imaging • tomography • analys...,945
5,5,Macro 5: chart • bar charts • scatterplots • p...,602
8,8,Macro 8: multidimensional scaling • scatterplo...,543
0,0,Macro 0: direct volume rendering • volume rend...,317
3,3,Macro 3: flow field • flow fields • computatio...,313
4,4,Macro 4: volume rendering • molecular modeling...,245
6,6,Macro 6: volume rendering • sketching • design...,165
1,1,Macro 1: explainable machine learning • deep n...,153
7,7,Macro 7: weather forecasting • weather predict...,126
9,9,Macro 9: sensemaking process • sensemaking • p...,122
